# CIDS · 00 — Generate narrative (Type-N) prompts

**Flow:** local project → commit + push → GitHub `Vemuri-BK/cids` (public) → clone here → run.  
**Settings:** Accelerator = GPU T4 x2 (or P100), Internet = ON.  
**Inputs:** attach the `cids-assets` dataset (contains `cids_prompts.csv`).

Run 1 (all 1506 cases, Qwen2.5-7B) → Run 2 (306 test cases, Phi-3.5-mini, held-out wording). Both runs are resumable: if the session stops, just run the notebook again.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'   # public repo, no token needed
GIT_BRANCH          = 'main'
# Pin the exact commit you pushed (git log -1 --format=%H). Empty = use latest on the branch.
EXPECTED_GIT_COMMIT = ''

# Kaggle dataset owner slug is 'bharathkumarvemuri'
PROMPTS_CSV = '/kaggle/input/datasets/bharathkumarvemuri/cids-assets/cids_prompts.csv'
CODE_DIR    = '/kaggle/working/cids'
WORK        = '/kaggle/working'
OUT         = f'{WORK}/narratives'

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q -U bitsandbytes sentence-transformers accelerate
!pytest -q tests

In [ ]:
from pathlib import Path
if not Path(PROMPTS_CSV).exists():   # fall back to auto-detect if the mount path differs
    found = list(Path('/kaggle/input').rglob('cids_prompts.csv'))
    assert len(found) == 1, f'Set PROMPTS_CSV explicitly. Found: {found}'
    PROMPTS_CSV = str(found[0])
PROMPTS = PROMPTS_CSV
print(PROMPTS)

## Smoke test (16 cases) — check the printed pass rates before the full run

In [ ]:
!python scripts/generate_narratives.py --prompts {PROMPTS} --target train_style --out_dir {OUT}/smoke --limit 16

In [ ]:
import json
recs = [json.loads(l) for l in open(f'{OUT}/smoke/train_style_progress.jsonl')]
for r in recs[:3]:
    print(r['patient_id'], r['status'])
    for t in r['chosen']: print('   ', t)
    bad = [c for c in r['candidates'] if c['problems']]
    if bad: print('   rejected example:', bad[0]['problems'])

## Run 1 — train-style narratives, all cases (~1–3 h on T4)

In [ ]:
!python scripts/generate_narratives.py --prompts {PROMPTS} --target train_style --out_dir {OUT}

## Run 2 — held-out narratives from a different LLM, test cases only

In [ ]:
!python scripts/generate_narratives.py --prompts {OUT}/cids_prompts_with_train_style.csv --target heldout --out_dir {OUT}

In [ ]:
import shutil
shutil.copy(f'{OUT}/cids_prompts_with_heldout.csv', f'{WORK}/cids_prompts_final.csv')
import pandas as pd
d = pd.read_csv(f'{WORK}/cids_prompts_final.csv', keep_default_na=False)
print({c: int((d[c].str.len()>0).sum()) for c in d.columns if c.startswith('narrative')})
# Download cids_prompts_final.csv from the Output panel, then locally:
# python scripts/update_excel.py --csv cids_prompts_final.csv --xlsx data/prompts/cids_prompts.xlsx